In [9]:
# Designing a daily, genre-level dataset that enables stakeholders to understand audience activity, content performance, and satisfaction.

Phase 3: Business-Ready Metrics (Gold Layer)
The Goal: Create the "Data Product" that stakeholders actually use for decisions.
I'm transforming individual rows into high-level metrics. This layer is optimized for speed and clarity, so Looker dashboards load quickly for staleholders.

The Goal: Finalize the "Data Product" by creating dense, actionable metrics for executive-level reporting.

Intelligent Data Expansion: My initial profiling revealed a "sparse" dataset (low event density), which can lead to misleading or "choppy" visualizations. To solve this, I’ve introduced a Synthetic Data Expansion step.
Multi-Grain Time Series: I'm aggregating engagement at the Daily, Weekly, and Monthly levels.
Daily/weekly helps us spot viral content quickly and pivot rapidly with what content to feature.
Monthly reveals the long-term or seasonal "shifts" in audience taste to inform future budgets and programming decisions.


In [10]:
# loading the silver interactions table
import pandas as pd

df_silver = pd.read_parquet("../data/03_silver/silver_interactions.parquet")
pd.set_option('display.width', 300)
print(df_silver.head())

   userId  movieId  rating    rating_timestamp                   title     genres
0       1       31     2.5 2009-12-14 02:52:24  Dangerous Minds (1995)      Drama
1       1     1029     3.0 2009-12-14 02:52:59            Dumbo (1941)  Animation
2       1     1029     3.0 2009-12-14 02:52:59            Dumbo (1941)   Children
3       1     1029     3.0 2009-12-14 02:52:59            Dumbo (1941)      Drama
4       1     1029     3.0 2009-12-14 02:52:59            Dumbo (1941)    Musical


By duplicating and "perturbing" (slightly varying) existing interactions, I’ve created a more robust dataset that mimics high-volume traffic. This ensures that downstream aggregations are statistically significant and realistic for a production-scale dashboard.

In [11]:
# Scale.
MULTIPLIER = 100  # 10–100 is good

# Create synthetic copies
import numpy as np

df_list = []

for i in range(MULTIPLIER):
    temp = df_silver.copy()
    
    # shift user IDs so they don’t collide
    temp["userId"] = temp["userId"] + (i * 100000)
    
    # add random time offset (within ~30 days and 2 hours) to rating timestamps to spread out interactions
    temp["rating_timestamp"] = temp["rating_timestamp"] + pd.to_timedelta(
        np.random.randint(-15, 15, size=len(temp)), unit="D") + pd.to_timedelta(
            np.random.randint(-60, 60, size=len(temp)), unit="m")
    
    
    # add rating noise for fun (and to prevent perfect duplicates)
    temp["rating"] = temp["rating"] + np.random.normal(0, 0.2, size=len(temp))
    temp["rating"] = temp["rating"].clip(0.5, 5.0)
    
    df_list.append(temp)

df_big = pd.concat(df_list, ignore_index=True)

print(df_big.head())





   userId  movieId    rating    rating_timestamp                   title     genres
0       1       31  2.663803 2009-12-15 03:36:24  Dangerous Minds (1995)      Drama
1       1     1029  3.181078 2009-12-27 03:09:59            Dumbo (1941)  Animation
2       1     1029  3.164562 2009-12-24 02:23:59            Dumbo (1941)   Children
3       1     1029  3.076958 2009-12-24 02:30:59            Dumbo (1941)      Drama
4       1     1029  2.879660 2009-12-25 02:30:59            Dumbo (1941)    Musical



Metrics Definition
active_users: Number of unique users interacting with content in a given day, week, month
nunique(user_id)
engagement_events: Total number of rating events (using this as a proxy for user engagement)
count(rating)
avg_rating: Average rating given to content (using this as a proxy for user satisfaction)
mean(rating)
high_rating_pct: percentage of ratings equal to or above 4.0 (using this as an approximation of positive sentiment)
(rating >= 4).mean()
unique_titles_engaged: sum of unique movies
nunique(title)


In [12]:
# Add date and hour columns for easier aggregation in the gold table. The date column allows for analysis of daily trends, while the hour column will help us understand how engagement varies throughout the day.
df_big["date"] = df_big["rating_timestamp"].dt.floor("D")
df_big["hour"] = df_big["rating_timestamp"].dt.hour

# gold table implementation 

gold = df_big.groupby(["date", "hour", "genres"]).agg(
    active_users=("userId", "nunique"),
    engagement_events=("rating", "count"),
    avg_rating=("rating", "mean")
).reset_index()

print(gold.head())

        date  hour    genres  active_users  engagement_events  avg_rating
0 1994-12-25    10    Comedy             4                  4    3.145263
1 1994-12-25    10     Crime             2                  2    3.097898
2 1994-12-25    10  Thriller             1                  1    5.000000
3 1994-12-25    11    Comedy             6                  6    2.844917
4 1994-12-25    11     Crime             5                  5    2.911648


In [13]:
gold.to_parquet("../data/04_gold/audience_engagement_daily.parquet", index=False)